# 실험2 — MOSAIC 구성요소 분해 (ACM2 기준)

MOSAIC = **carry ⊕ overlap**. 둘을 분리해 각 기여를 본다 (2 seed, LIBERO-10).

| 모델 | carry | overlap | 학습 |
|---|:--:|:--:|---|
| `acm2` | ✗ | ✗ | 기존 (재사용) |
| `acm2_carry` | ✓ | ✗ | **새 학습** (carry가 가중치를 바꿈) |
| `acm2_overlap` | ✗ | ✓ | acm2 재해석 (재학습 X) |
| `acm2_mosaic` | ✓ | ✓ | acm2_carry 재해석 (재학습 X) |

overlap은 추론 전용이라 `acm2_overlap`/`acm2_mosaic`은 재학습이 필요 없다.
**유일한 새 학습 = `acm2_carry`** (그리고 그게 `acm2_mosaic`의 소스).


## 0) 부팅 + 모델 목록


In [ ]:
import sys, json
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)
v23 = cf.v23

TASK  = 'libero_10'          # 대표 benchmark 1개 (교수님 지시)
SEEDS = [0, 1]               # 추가실험 = 2 seed
# 여러 노드로 쪼갤 때만 사용: 각 노드에서 NODE=0,1,.. / NNODES=노드수. 기본=단일 노드.
NODE, NNODES = 0, 1

TAGS = ['acm2', 'acm2_carry', 'acm2_overlap', 'acm2_mosaic']
GPUS = v23.available_gpus()  # 이 노드의 GPU (보통 2개)
print('실험2 — MOSAIC 구성요소 분해 (carry ⊕ overlap)')
print('task', TASK, '| seeds', SEEDS, '| GPU', GPUS)
for t in TAGS:
    print(f'  {t:16} {v23.MODEL_LABELS.get(t, t)}')

## 1) 상태 점검


In [ ]:
# 상태: 각 모델의 '소스 체크포인트' 유무 + eval 완료 여부.
#   overlap 태그는 base(act/acm2/acm2_carry)를, *_te 는 base(act/bimamba)를 재사용한다.
def src_of(t):
    if t in cf.OVERLAP_SOURCE: return cf.OVERLAP_SOURCE[t]   # overlap 재해석 소스
    if t.endswith('_te'):      return t[:-3]                 # TE 는 base ckpt 재사용
    return t                                                # 직접

_MINEP = 10 * cf.EVAL_N_EP // 2      # 유효 eval 기준: overall n_ep >= 2500
def eval_done(t, s):
    info = cf.eval_rep_dir(t, s, TASK, 0, cf.CKPT_STEP) / 'eval_info.json'
    if not info.exists(): return False
    try:
        ov = json.loads(info.read_text()).get('overall', {})
        return (ov.get('n_ep', ov.get('n_episodes')) or 0) >= _MINEP
    except Exception:
        return False

rows = []
for t in TAGS:
    src = src_of(t)
    kind = ('overlap재해석' if t in cf.OVERLAP_SOURCE else
            'TE(eval전용)' if t.endswith('_te') else '직접')
    for s in SEEDS:
        has = v23.best_ckpt_dir(src, s, TASK, how=cf.CKPT_STEP) is not None
        done = eval_done(t, s)
        act = ('eval완료' if done else
               ('eval필요' if has else f'소스[{src}] 없음 → 학습 필요'))
        rows.append((t, s, kind, src, '있음' if has else '없음', '완료' if done else '—', act))
print(f'{"tag":16}{"seed":5}{"방식":14}{"소스":14}{"소스ckpt":9}{"eval":6}action')
for r in rows:
    print(f'{r[0]:16}{r[1]:<5}{r[2]:14}{r[3]:14}{r[4]:9}{r[5]:6}{r[6]}')
need = sorted({r[3] for r in rows if r[4] == '없음'})
print('\n⚠️ 소스 체크포인트 없어 학습 필요:', need) if need else \
    print('\n✅ 모든 소스 존재 → 재학습 없이 eval 만 하면 됨')

## 2) `acm2_carry` 학습 (실험2 유일한 새 학습 — 이미 있으면 자동 skip)


In [ ]:
# (실험2 전용) acm2_carry = carry ON → 진짜 새 모델이라 학습 필요. acm2_mosaic 의 소스이기도 함.
#   ⚠️ 이 셀만 학습(GPU 오래 씀). 나머지 실험2 모델(acm2·acm2_overlap·acm2_mosaic)은 재학습 없음.
#   이미 150k 학습됐으면 run_training_jobs 가 자동 skip. 재학습 정말 피하려면 이 셀만 건너뛰면 되고,
#   그러면 acm2_carry·acm2_mosaic 두 칸은 비게 된다(overlap 분해 표가 절반만 채워짐).
cf.run_training_jobs([('acm2_carry', s, TASK) for s in SEEDS], GPUS, prefetch_task=TASK)

## 3) overlap 재해석 (`acm2_overlap` ← acm2, `acm2_mosaic` ← acm2_carry)


In [ ]:
# overlap 재해석 (재학습 X): base 체크포인트 복사 + config.json 의 type/sscp_overlap 만 패치.
#   overlap 정책은 파라미터 0개 추가 subclass → base 가중치를 그대로 사용(안전).
#   base 없으면 그 태그는 생략(경고). *_te·직접 태그는 여기서 아무것도 안 함.
prepared = cf.prepare_overlap_ckpts(TAGS, SEEDS, TASK)
print('\n재해석 완료:', sum(1 for _, _, pm in prepared if pm is not None),
      '/', len(prepared), '(base 없어 생략된 건 위 경고 참고)')

## 4) eval 실행 (1 rep × 500ep × 10 task)


In [ ]:
# eval — overlap 재해석 ckpt + TE(eval전용) + 직접, 전부 같은 기계(run_libero_eval_jobs).
#   기본 1 rep × 500ep (= 500×10task = 5,000 에피소드, 유효 기준 충족). 이미 끝난 run 은 skip.
#   여러 노드로 쪼갤 땐 위 boot 에서 NODE/NNODES 만 바꿔 각 노드에서 실행.
pairs = [(t, s) for t in TAGS for s in SEEDS]
if NNODES > 1:
    pairs = pairs[NODE::NNODES]
    print(f'노드 {NODE}/{NNODES} 담당 {len(pairs)} 쌍:', pairs)
cf.run_libero_eval_jobs(pairs, GPUS, task=TASK, n_episodes=cf.EVAL_N_EP, reps=[0])